In [ ]:
!pip -q install nibabel scipy scikit-learn tqdm kagglehub psutil

In [ ]:
import os
import re
import gc
import json
import math
import time
import ctypes
import random
import shutil
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple, Any

import psutil
import numpy as np
import nibabel as nib
from scipy import ndimage
from sklearn.model_selection import train_test_split, KFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Cố định Random Seed = 42 để tái lập chuẩn xác 100%
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

try:
    _LIBC = ctypes.CDLL("libc.so.6")
except Exception:
    _LIBC = None

def trim_os_memory():
    gc.collect()
    if _LIBC is not None:
        try:
            _LIBC.malloc_trim(0)
        except Exception:
            pass

def get_ram_status() -> str:
    vm = psutil.virtual_memory()
    used_gb = vm.used / (1024 ** 3)
    total_gb = vm.total / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU VRAM: {gpu_alloc:.2f} GB"
    return f"System RAM: {used_gb:.2f}/{total_gb:.2f} GB ({vm.percent:.1f}%){gpu_str}"

def clear_memory():
    trim_os_memory()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Đường dẫn cấu hình cục bộ hoàn toàn (/content)
BASE_WEIGHTS_PATH = Path("/content/3dunet_final_fp32.pth")
if not BASE_WEIGHTS_PATH.exists():
    alt_weights = Path("/content/unet3d_final_fp32.pth")
    if alt_weights.exists():
        BASE_WEIGHTS_PATH = alt_weights

LOCAL_CACHE_DIR = Path("/content/unet3d_continual_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

BRATS2021_MANIFEST_PATH = Path("/content/brats2021_50_manifest.json")
FINETUNED_WEIGHTS_PATH = Path("/content/unet3d_finetuned_fp32.pth")
OPCM_WEIGHTS_PATH = Path("/content/unet3d_opcm_merged_fp32.pth")
PROD_WEIGHTS_PATH = Path("/content/unet3d_production_wise_ft.pth")
PROD_CALIB_PATH = Path("/content/calibration_cache.json")

# Siêu tham số kiến trúc & huấn luyện
BASE_FEATURES = (16, 32, 64, 128)
BOTTLENECK_FEATURES = 256
DROPOUT = 0.20

PATCH_SIZE = (128, 128, 128)
BATCH_SIZE = 2
SAMPLES_PER_VOLUME = 4
FT_EPOCHS = 15
FT_LR = 3e-5
VAL_EVERY = 5
USE_AMP = True
NUM_WORKERS = min(2, os.cpu_count() or 1)

MC_DROPOUT_PASSES = 20
VOXEL_VOLUME_CM3 = 1.0 / 1000.0  # 1mm x 1mm x 1mm -> cm^3

print(f"Device: {device} | {get_ram_status()}")
print(f"Base weights path: {BASE_WEIGHTS_PATH} (Exists: {BASE_WEIGHTS_PATH.exists()})")

Device: cuda | System RAM: 1.91/52.96 GB (4.8%) | GPU VRAM: 0.00 GB
Base weights path: /content/unet3d_final_fp32.pth (Exists: True)


In [ ]:
class ResBlock3D(nn.Module):
    """
    3D Residual Convolutional Block (Milletari et al., 2016; Çiçek et al., 2016)
    """
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_ch, affine=True)
        self.relu = nn.LeakyReLU(negative_slope=0.01, inplace=True)
        self.drop = nn.Dropout3d(dropout)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_ch, affine=True)

        if in_ch != out_ch:
            self.skip = nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.relu(self.norm1(self.conv1(x)))
        out = self.drop(out)
        out = self.norm2(self.conv2(out))
        out = self.relu(out + residual)
        return out


class ResidualUNet3D(nn.Module):
    """
    Baseline 3D Residual U-Net for BraTS:
    Class order: [0: BG, 1: ET, 2: NCR/NET, 3: ED]
    """
    def __init__(self, in_channels=4, num_classes=4, features=(16, 32, 64, 128), bottleneck=256, dropout=0.2):
        super().__init__()
        f1, f2, f3, f4 = features

        self.stem = nn.Conv3d(in_channels, f1, kernel_size=3, padding=1, bias=False)
        self.enc1 = ResBlock3D(f1, f1, dropout)
        self.down1 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc2 = ResBlock3D(f1, f2, dropout)
        self.down2 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc3 = ResBlock3D(f2, f3, dropout)
        self.down3 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc4 = ResBlock3D(f3, f4, dropout)
        self.down4 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.bottleneck = ResBlock3D(f4, bottleneck, dropout)

        self.up4 = nn.ConvTranspose3d(bottleneck, f4, kernel_size=2, stride=2)
        self.dec4 = ResBlock3D(f4 + f4, f4, dropout)

        self.up3 = nn.ConvTranspose3d(f4, f3, kernel_size=2, stride=2)
        self.dec3 = ResBlock3D(f3 + f3, f3, dropout)

        self.up2 = nn.ConvTranspose3d(f3, f2, kernel_size=2, stride=2)
        self.dec2 = ResBlock3D(f2 + f2, f2, dropout)

        self.up1 = nn.ConvTranspose3d(f2, f1, kernel_size=2, stride=2)
        self.dec1 = ResBlock3D(f1 + f1, f1, dropout)

        self.head = nn.Conv3d(f1, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        e4 = self.enc4(self.down3(e3))

        b = self.bottleneck(self.down4(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)


def enable_mc_dropout(model: nn.Module) -> None:
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):
            m.train()


if not BASE_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file trọng số tại {BASE_WEIGHTS_PATH}. Vui lòng tải file lên /content/ trước!")

raw_ckpt = torch.load(str(BASE_WEIGHTS_PATH), map_location="cpu", weights_only=False)
base_state_dict = raw_ckpt["model_state_dict"] if (isinstance(raw_ckpt, dict) and "model_state_dict" in raw_ckpt) else raw_ckpt
SAVED_POSTPROCESS = raw_ckpt.get("postprocess", {"second_lcc_min_voxels": 500, "et_min_voxels": 50}) if isinstance(raw_ckpt, dict) else {"second_lcc_min_voxels": 500, "et_min_voxels": 50}

theta_0 = {k: v.detach().cpu().clone() for k, v in base_state_dict.items()}
del raw_ckpt, base_state_dict
clear_memory()

model = ResidualUNet3D(
    in_channels=4,
    num_classes=4,
    features=BASE_FEATURES,
    bottleneck=BOTTLENECK_FEATURES,
    dropout=DROPOUT
).to(device)
model.load_state_dict(theta_0)

total_params = sum(p.numel() for p in model.parameters())
print(f"Đã load thành công trọng số theta_0 từ: {BASE_WEIGHTS_PATH}")
print(f"Total params: {total_params:,} ({total_params * 4 / 1024**2:.2f} MiB) | Post-process config: {SAVED_POSTPROCESS}")
print(get_ram_status())

Đã load thành công trọng số theta_0 từ: /content/unet3d_final_fp32.pth
Total params: 5,741,684 (21.90 MiB) | Post-process config: {'second_lcc_min_voxels': 500, 'et_min_voxels': 50, 'WT': 0.9025704510449865, 'TC': 0.8742188267930782, 'ET': 0.8004246162910961, 'Mean': 0.8590712980430536}
System RAM: 2.04/52.96 GB (5.0%) | GPU VRAM: 0.02 GB


In [ ]:
import tarfile
import zipfile
import kagglehub

# 1. Tải BraTS 2020
path_2020 = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
print("Path to BraTS 2020 files:", path_2020)

# 2. Tải BraTS 2021 Task 1
path_2021 = kagglehub.dataset_download("dschettler8845/brats-2021-task1")
print("Path to BraTS 2021 files:", path_2021)

DATA_ROOT_2020 = Path(path_2020)
RAW_ROOT_2021 = Path(path_2021)

# Một số phiên bản Kaggle BraTS 2021 đóng gói file .tar bên trong thư mục tải về -> giải nén cục bộ nếu cần
EXTRACTED_2021_DIR = Path("/content/brats2021_extracted")
tar_files_2021 = sorted(RAW_ROOT_2021.rglob("*.tar")) + sorted(RAW_ROOT_2021.rglob("*.tar.gz"))
if len(tar_files_2021) > 0:
    EXTRACTED_2021_DIR.mkdir(parents=True, exist_ok=True)
    for tf in tar_files_2021:
        marker = EXTRACTED_2021_DIR / f".done_{tf.name}"
        if not marker.exists():
            print(f"Đang giải nén archive BraTS 2021: {tf.name} -> {EXTRACTED_2021_DIR} ...")
            with tarfile.open(tf, "r:*") as tar:
                tar.extractall(path=EXTRACTED_2021_DIR)
            marker.touch()
    DATA_ROOT_2021 = EXTRACTED_2021_DIR
else:
    DATA_ROOT_2021 = RAW_ROOT_2021


def find_one(case_dir: Path, suffix: str):
    candidates = list(case_dir.glob("*.nii")) + list(case_dir.glob("*.nii.gz"))
    suffix = suffix.lower()
    for p in candidates:
        name = p.name.lower()
        if name.endswith(f"_{suffix}.nii") or name.endswith(f"_{suffix}.nii.gz"):
            return p
    return None


def discover_cases(root: Path, dataset_prefix: str) -> List[dict]:
    cases = []
    for d in sorted(root.rglob("*")):
        if not d.is_dir():
            continue
        flair = find_one(d, "flair")
        t1 = find_one(d, "t1")
        t1ce = find_one(d, "t1ce")
        t2 = find_one(d, "t2")
        seg = find_one(d, "seg")
        if all([flair, t1, t1ce, t2, seg]):
            cases.append({
                "id": d.name,
                "full_id": f"{dataset_prefix}__{d.name}",
                "flair": str(flair),
                "t1": str(t1),
                "t1ce": str(t1ce),
                "t2": str(t2),
                "seg": str(seg),
            })
    return cases


cases_2020 = discover_cases(DATA_ROOT_2020, "brats2020")
cases_2021_all = discover_cases(DATA_ROOT_2021, "brats2021")

# Tái lập y hệt tập Train (294) và Golden Validation (74) của BraTS 2020 với SEED = 42
case_ids_2020 = [c["id"] for c in cases_2020]
train_ids_2020, val_ids_2020 = train_test_split(
    case_ids_2020,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)
train_ids_set = set(train_ids_2020)
val_ids_set = set(val_ids_2020)

train_cases_2020 = [c for c in cases_2020 if c["id"] in train_ids_set]
val_cases_2020 = [c for c in cases_2020 if c["id"] in val_ids_set]

print(f"Tổng BraTS 2020: {len(cases_2020)} | Train gốc: {len(train_cases_2020)} | Golden Validation: {len(val_cases_2020)}")
print(f"Tổng BraTS 2021 tìm thấy trên local: {len(cases_2021_all)} ca | {get_ram_status()}")

Using Colab cache for faster access to the 'brats20-dataset-training-validation' dataset.
Path to BraTS 2020 files: /kaggle/input/brats20-dataset-training-validation
Using Colab cache for faster access to the 'brats-2021-task1' dataset.
Path to BraTS 2021 files: /kaggle/input/brats-2021-task1
Đang giải nén archive BraTS 2021: BraTS2021_00495.tar -> /content/brats2021_extracted ...
Đang giải nén archive BraTS 2021: BraTS2021_00621.tar -> /content/brats2021_extracted ...


/tmp/ipykernel_392/3738675099.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACTED_2021_DIR)


Đang giải nén archive BraTS 2021: BraTS2021_Training_Data.tar -> /content/brats2021_extracted ...
Tổng BraTS 2020: 368 | Train gốc: 294 | Golden Validation: 74
Tổng BraTS 2021 tìm thấy trên local: 1251 ca | System RAM: 2.15/52.96 GB (5.3%) | GPU VRAM: 0.02 GB


In [ ]:
def compute_volume_fingerprint(flair_path: str) -> str:
    """
    Tạo chữ ký định danh (fingerprint) dựa trên thống kê hình học & cường độ voxel non-zero
    của chuỗi xung FLAIR (không phụ thuộc tên file BraTS20_Training_xxx vs BraTS2021_0xxxx)
    để phát hiện chính xác các ca BraTS 2021 trùng lặp với BraTS 2020.
    """
    nii = nib.load(flair_path)
    arr = np.asarray(nii.dataobj, dtype=np.float32)
    nz = arr[arr != 0]
    if nz.size == 0:
        sig = f"{arr.shape}_0"
    else:
        p10, p50, p90, p95 = np.percentile(nz, [10.0, 50.0, 90.0, 95.0])
        sig = (
            f"{arr.shape}_{nz.size}_"
            f"{p10:.3f}_{p50:.3f}_{p90:.3f}_{p95:.3f}_"
            f"{float(nz.mean()):.3f}_{float(nz.std()):.3f}"
        )
    del arr, nz, nii
    return hashlib.sha256(sig.encode("utf-8")).hexdigest()[:24]


def p95_normalize(vol: np.ndarray) -> np.ndarray:
    vol = vol.astype(np.float32, copy=False)
    nz = vol[vol != 0]
    if nz.size == 0:
        return np.zeros_like(vol, dtype=np.float32)
    p95 = np.percentile(nz, 95.0)
    out = np.zeros_like(vol, dtype=np.float32)
    if p95 > 0:
        mask = vol != 0
        out[mask] = vol[mask] / p95
    return out


def remap_seg_brats(seg: np.ndarray) -> np.ndarray:
    """
    Chuẩn hóa nhãn phân vùng BraTS 2020 & BraTS 2021 về định dạng ResidualUNet3D:
    0: Background | 1: ET (gốc 4) | 2: NCR/NET (gốc 1) | 3: ED (gốc 2)
    """
    out = np.zeros(seg.shape, dtype=np.uint8)
    out[seg == 4] = 1
    out[seg == 1] = 2
    out[seg == 2] = 3
    return out


def pad_to_min_shape(image: np.ndarray, target: np.ndarray, min_shape=(128, 128, 128)):
    _, D, H, W = image.shape
    pD = max(0, min_shape[0] - D)
    pH = max(0, min_shape[1] - H)
    pW = max(0, min_shape[2] - W)
    if pD > 0 or pH > 0 or pW > 0:
        image = np.pad(image, ((0, 0), (0, pD), (0, pH), (0, pW)), mode="constant", constant_values=0)
        target = np.pad(target, ((0, pD), (0, pH), (0, pW)), mode="constant", constant_values=0)
    return image, target


def cache_nifti_case_to_npy(case: dict, cache_dir: Path, force: bool = False) -> dict:
    cid = case["full_id"]
    image_path = cache_dir / f"{cid}_image_3d.npy"
    seg_path = cache_dir / f"{cid}_seg_3d.npy"
    meta_path = cache_dir / f"{cid}_meta_3d.json"

    if image_path.exists() and seg_path.exists() and meta_path.exists() and not force:
        return json.loads(meta_path.read_text())

    vols = []
    for key in ["flair", "t1", "t1ce", "t2"]:
        vol = nib.load(case[key]).get_fdata(dtype=np.float32)
        vols.append(p95_normalize(vol))
    seg = remap_seg_brats(nib.load(case["seg"]).get_fdata(dtype=np.float32))

    image = np.stack(vols, axis=0).transpose(0, 3, 1, 2)
    target = seg.transpose(2, 0, 1)
    image, target = pad_to_min_shape(image, target, PATCH_SIZE)

    np.save(image_path, image.astype(np.float16))
    np.save(seg_path, target.astype(np.uint8))

    flat_fg = np.flatnonzero(target.ravel())
    if flat_fg.size > 2500:
        step = max(1, flat_fg.size // 2500)
        flat_fg = flat_fg[::step]
    if flat_fg.size > 0:
        coords_tuple = np.unravel_index(flat_fg, target.shape)
        fg_indices = np.stack(coords_tuple, axis=1).astype(np.int16).tolist()
    else:
        fg_indices = []

    counts = np.bincount(target.ravel(), minlength=4).astype(np.int64).tolist()
    meta = {
        "case_id": cid,
        "raw_id": case["id"],
        "image_path": str(image_path),
        "seg_path": str(seg_path),
        "shape": list(target.shape),
        "fg_coords": fg_indices,
        "class_pixel_counts": counts,
    }
    meta_path.write_text(json.dumps(meta))
    del vols, seg, image, target
    return meta


# ==============================================================================
# BƯỚC 1: KIỂM TRA TRÙNG LẶP DỮ LIỆU GIỮA BRATS 2021 VÀ BRATS 2020
# ==============================================================================
print("=== [1/3] KIỂM TRA TRÙNG LẶP GIỮA BRATS 2020 VÀ BRATS 2021 ===")
fp_to_2020_train = {}
fp_to_2020_val = {}

for idx, c in enumerate(tqdm(train_cases_2020, desc="Fingerprint BraTS2020 Train (294)")):
    fp = compute_volume_fingerprint(c["flair"])
    fp_to_2020_train[fp] = c["id"]
    if (idx + 1) % 50 == 0:
        trim_os_memory()

for idx, c in enumerate(tqdm(val_cases_2020, desc="Fingerprint Golden Val (74)")):
    fp = compute_volume_fingerprint(c["flair"])
    fp_to_2020_val[fp] = c["id"]
    if (idx + 1) % 25 == 0:
        trim_os_memory()

unique_brats2021_candidates = []
dup_in_train_2020 = []
dup_in_val_2020 = []

for idx, c in enumerate(tqdm(cases_2021_all, desc="Đối chiếu loại trùng BraTS 2021")):
    fp = compute_volume_fingerprint(c["flair"])
    if fp in fp_to_2020_val:
        dup_in_val_2020.append((c["id"], fp_to_2020_val[fp]))
    elif fp in fp_to_2020_train:
        dup_in_train_2020.append((c["id"], fp_to_2020_train[fp]))
    else:
        c["fingerprint"] = fp
        unique_brats2021_candidates.append(c)
    if (idx + 1) % 100 == 0:
        trim_os_memory()

trim_os_memory()
print("-" * 70)
print(f"Tổng số ca BraTS 2021 quét được:                   {len(cases_2021_all)}")
print(f"Số ca BraTS 2021 TRÙNG với BraTS 2020 Train:      {len(dup_in_train_2020)} ca (Đã loại)")
print(f"Số ca BraTS 2021 TRÙNG với BraTS 2020 Golden Val: {len(dup_in_val_2020)} ca (Đã loại - Chống Data Leakage!)")
print(f"Số ca BraTS 2021 MỚI HOÀN TOÀN (Không trùng):     {len(unique_brats2021_candidates)} ca")
if len(dup_in_val_2020) > 0:
    print(f"  Ví dụ ca trùng Val đã chặn: {dup_in_val_2020[:3]}")
print("-" * 70)

if len(unique_brats2021_candidates) < 50:
    raise RuntimeError(f"Chỉ còn {len(unique_brats2021_candidates)} ca BraTS 2021 không trùng (< 50)!")

# ==============================================================================
# BƯỚC 2: CHỌN ĐÚNG 50 CA BRATS 2021 (SEED=42) & LƯU MANIFEST ĐỂ REPRODUCE
# ==============================================================================
if BRATS2021_MANIFEST_PATH.exists():
    saved_manifest = json.loads(BRATS2021_MANIFEST_PATH.read_text(encoding="utf-8"))
    saved_ids_set = set(saved_manifest["selected_brats2021_ids"])
    selected_brats2021_50 = [c for c in unique_brats2021_candidates if c["id"] in saved_ids_set]
    print(f"Đã nạp danh sách 50 ca BraTS 2021 từ manifest có sẵn: {BRATS2021_MANIFEST_PATH}")
else:
    rng_2021 = random.Random(SEED)
    selected_brats2021_50 = sorted(
        rng_2021.sample(unique_brats2021_candidates, 50),
        key=lambda x: x["id"]
    )
    manifest_payload = {
        "seed": SEED,
        "source_dataset_2020": "awsaf49/brats20-dataset-training-validation",
        "source_dataset_2021": "dschettler8845/brats-2021-task1",
        "deduplication_summary": {
            "total_brats2021_scanned": len(cases_2021_all),
            "duplicates_with_brats2020_train": len(dup_in_train_2020),
            "duplicates_with_brats2020_val": len(dup_in_val_2020),
            "unique_brats2021_available": len(unique_brats2021_candidates),
        },
        "brats2020_train_ids": sorted(train_ids_2020),
        "brats2020_golden_val_ids": sorted(val_ids_2020),
        "selected_brats2021_count": len(selected_brats2021_50),
        "selected_brats2021_ids": [c["id"] for c in selected_brats2021_50],
        "selected_brats2021_records": [
            {"id": c["id"], "full_id": c["full_id"], "fingerprint": c.get("fingerprint", "")}
            for c in selected_brats2021_50
        ],
    }
    BRATS2021_MANIFEST_PATH.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")
    print(f"Đã lưu danh sách 50 ca BraTS 2021 & splits manifest tại: {BRATS2021_MANIFEST_PATH}")

# ==============================================================================
# BƯỚC 3: PREPROCESS SANG .NPY CỤC BỘ (294 BRATS2020 + 50 BRATS2021 + 74 VAL)
# ==============================================================================
print("\n=== [2/3] PREPROCESS DỮ LIỆU SANG .NPY LOCAL ===")
train_metas_2020 = []
for idx, c in enumerate(tqdm(train_cases_2020, desc="Preprocess BraTS2020 Train (294)")):
    train_metas_2020.append(cache_nifti_case_to_npy(c, LOCAL_CACHE_DIR))
    if (idx + 1) % 30 == 0:
        trim_os_memory()

val_metas_golden = []
for idx, c in enumerate(tqdm(val_cases_2020, desc="Preprocess Golden Val (74)")):
    val_metas_golden.append(cache_nifti_case_to_npy(c, LOCAL_CACHE_DIR))
    if (idx + 1) % 25 == 0:
        trim_os_memory()

train_metas_2021_50 = []
for idx, c in enumerate(tqdm(selected_brats2021_50, desc="Preprocess 50 BraTS2021 Mới (50)")):
    train_metas_2021_50.append(cache_nifti_case_to_npy(c, LOCAL_CACHE_DIR))
    if (idx + 1) % 10 == 0:
        trim_os_memory()

merged_train_metas = train_metas_2020 + train_metas_2021_50
clear_memory()

print("\n" + "=" * 70)
print(f"Tập Train gốc (BraTS 2020):          {len(train_metas_2020)} ca")
print(f"Tập bổ sung (BraTS 2021 lọc trùng):  {len(train_metas_2021_50)} ca")
print(f"Tổng tập Train sau khi gộp:          {len(merged_train_metas)} ca")
print(f"Tập Golden Validation (cố định):     {len(val_metas_golden)} ca")
print(f"File lưu danh sách reproduce:        {BRATS2021_MANIFEST_PATH}")
print(get_ram_status())

=== [1/3] KIỂM TRA TRÙNG LẶP GIỮA BRATS 2020 VÀ BRATS 2021 ===


Fingerprint BraTS2020 Train (294):   0%|          | 0/294 [00:00<?, ?it/s]

Fingerprint Golden Val (74):   0%|          | 0/74 [00:00<?, ?it/s]

Đối chiếu loại trùng BraTS 2021:   0%|          | 0/1251 [00:00<?, ?it/s]

----------------------------------------------------------------------
Tổng số ca BraTS 2021 quét được:                   1251
Số ca BraTS 2021 TRÙNG với BraTS 2020 Train:      285 ca (Đã loại)
Số ca BraTS 2021 TRÙNG với BraTS 2020 Golden Val: 71 ca (Đã loại - Chống Data Leakage!)
Số ca BraTS 2021 MỚI HOÀN TOÀN (Không trùng):     895 ca
  Ví dụ ca trùng Val đã chặn: [('BraTS2021_00107', 'BraTS20_Training_251'), ('BraTS2021_00110', 'BraTS20_Training_230'), ('BraTS2021_00112', 'BraTS20_Training_232')]
----------------------------------------------------------------------
Đã lưu danh sách 50 ca BraTS 2021 & splits manifest tại: /content/brats2021_50_manifest.json

=== [2/3] PREPROCESS DỮ LIỆU SANG .NPY LOCAL ===


Preprocess BraTS2020 Train (294):   0%|          | 0/294 [00:00<?, ?it/s]

Preprocess Golden Val (74):   0%|          | 0/74 [00:00<?, ?it/s]

Preprocess 50 BraTS2021 Mới (50):   0%|          | 0/50 [00:00<?, ?it/s]


Tập Train gốc (BraTS 2020):          294 ca
Tập bổ sung (BraTS 2021 lọc trùng):  50 ca
Tổng tập Train sau khi gộp:          344 ca
Tập Golden Validation (cố định):     74 ca
File lưu danh sách reproduce:        /content/brats2021_50_manifest.json
System RAM: 2.20/52.96 GB (5.3%) | GPU VRAM: 0.02 GB


In [ ]:
class FastBraTS3DPatchDataset(Dataset):
    """
    Đọc patch 3D trực tiếp qua numpy mmap_mode='r' từ SSD cục bộ -> System RAM gần như bằng 0.
    """
    def __init__(self, metas, patch_size=(128, 128, 128), samples_per_volume=4, fg_prob=0.67, augment=True):
        self.metas = metas
        self.patch_size = patch_size
        self.samples_per_volume = samples_per_volume
        self.fg_prob = fg_prob
        self.augment = augment
        self.total_samples = len(metas) * samples_per_volume

    def __len__(self):
        return self.total_samples

    def _crop_patch(self, img, seg, fg_coords):
        D, H, W = seg.shape
        pD, pH, pW = self.patch_size

        if random.random() < self.fg_prob and len(fg_coords) > 0:
            cD, cH, cW = fg_coords[random.randint(0, len(fg_coords) - 1)]
            sD = max(0, min(int(cD) - pD // 2, D - pD))
            sH = max(0, min(int(cH) - pH // 2, H - pH))
            sW = max(0, min(int(cW) - pW // 2, W - pW))
        else:
            sD = random.randint(0, max(0, D - pD))
            sH = random.randint(0, max(0, H - pH))
            sW = random.randint(0, max(0, W - pW))

        patch_img = img[:, sD:sD+pD, sH:sH+pH, sW:sW+pW]
        patch_seg = seg[sD:sD+pD, sH:sH+pH, sW:sW+pW]
        return patch_img, patch_seg

    def _augment(self, x, y):
        for dim in [1, 2, 3]:
            if random.random() < 0.5:
                x = torch.flip(x, dims=[dim])
                y = torch.flip(y, dims=[dim - 1])
        if random.random() < 0.5:
            scale = random.uniform(0.9, 1.1)
            shift = random.uniform(-0.05, 0.05)
            x = torch.where(x != 0, x * scale + shift, x)
        return x, y

    def __getitem__(self, idx):
        meta_idx = idx // self.samples_per_volume
        m = self.metas[meta_idx]

        img = np.load(m["image_path"], mmap_mode="r")
        seg = np.load(m["seg_path"], mmap_mode="r")

        patch_img, patch_seg = self._crop_patch(img, seg, m.get("fg_coords", []))
        x = torch.from_numpy(np.asarray(patch_img, dtype=np.float32).copy())
        y = torch.from_numpy(np.asarray(patch_seg, dtype=np.int64).copy())

        if self.augment:
            x, y = self._augment(x, y)
        return x, y


def compute_class_weights(metas) -> torch.Tensor:
    counts = np.zeros(4, dtype=np.float64)
    for m in metas:
        counts += np.asarray(m["class_pixel_counts"], dtype=np.float64)
    freq = counts / counts.sum()
    weights = 1.0 / np.sqrt(freq + 1e-12)
    weights /= weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def dice_ce_loss_3d(logits, target, class_weights=None, smooth=1e-5):
    ce = F.cross_entropy(logits, target.long(), weight=class_weights)
    num_classes = logits.shape[1]
    probs = torch.softmax(logits, dim=1)
    target_one_hot = F.one_hot(target.long(), num_classes=num_classes).permute(0, 4, 1, 2, 3).to(dtype=probs.dtype)

    dims = (0, 2, 3, 4)
    intersection = (probs * target_one_hot).sum(dim=dims)
    denominator = probs.sum(dim=dims) + target_one_hot.sum(dim=dims)

    dice_per_class = (2.0 * intersection + smooth) / (denominator + smooth)
    dice_loss = 1.0 - dice_per_class.mean()
    return dice_loss + ce


def postprocess_volume_3d(pred: np.ndarray, second_lcc_min_voxels: int = 500, et_min_voxels: int = 50) -> np.ndarray:
    pred = pred.copy()
    tumor = pred > 0
    labeled, n = ndimage.label(tumor)

    if n > 0:
        sizes = ndimage.sum(tumor, labeled, index=np.arange(1, n + 1))
        order = np.argsort(sizes)[::-1]
        keep_ids = [int(order[0] + 1)]
        if len(order) >= 2 and sizes[order[1]] >= second_lcc_min_voxels:
            keep_ids.append(int(order[1] + 1))
        keep_mask = np.isin(labeled, keep_ids)
        pred[~keep_mask] = 0

    et_count = int((pred == 1).sum())
    if 0 < et_count < et_min_voxels:
        pred[pred == 1] = 2
    return pred


@torch.no_grad()
def predict_case_3d_sliding_window(model: nn.Module, meta: dict, patch_size=(128, 128, 128), overlap=0.5) -> np.ndarray:
    model.eval()
    image = np.load(meta["image_path"], mmap_mode="r")
    C, D, H, W = image.shape
    pD, pH, pW = patch_size

    step_d = int(pD * (1 - overlap))
    step_h = int(pH * (1 - overlap))
    step_w = int(pW * (1 - overlap))

    steps_d = list(range(0, max(1, D - pD + 1), step_d))
    if steps_d[-1] + pD < D: steps_d.append(D - pD)
    steps_h = list(range(0, max(1, H - pH + 1), step_h))
    if steps_h[-1] + pH < H: steps_h.append(H - pH)
    steps_w = list(range(0, max(1, W - pW + 1), step_w))
    if steps_w[-1] + pW < W: steps_w.append(W - pW)

    prob_accum = np.zeros((4, D, H, W), dtype=np.float32)
    count_accum = np.zeros((D, H, W), dtype=np.float32)

    for sd in steps_d:
        for sh in steps_h:
            for sw in steps_w:
                sub_vol = image[:, sd:sd+pD, sh:sh+pH, sw:sw+pW]
                x = torch.from_numpy(np.asarray(sub_vol, dtype=np.float32)).unsqueeze(0).to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                    logits = model(x)
                    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
                prob_accum[:, sd:sd+pD, sh:sh+pH, sw:sw+pW] += probs
                count_accum[sd:sd+pD, sh:sh+pH, sw:sw+pW] += 1.0

    prob_accum /= np.maximum(count_accum, 1.0)
    pred_vol = prob_accum.argmax(axis=0).astype(np.uint8)
    del prob_accum, count_accum
    return pred_vol


def dice_binary(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denom)


def brats_metrics(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    wt = dice_binary(pred > 0, gt > 0)
    tc = dice_binary(np.isin(pred, [1, 2]), np.isin(gt, [1, 2]))
    et = dice_binary(pred == 1, gt == 1)
    return {"WT": wt, "TC": tc, "ET": et, "Mean": (wt + tc + et) / 3.0}


def evaluate_golden_val_dice(model: nn.Module, metas: List[dict], apply_pp: bool = True, desc: str = "Golden Val Dice") -> Dict[str, float]:
    rows = []
    lcc_min = SAVED_POSTPROCESS.get("second_lcc_min_voxels", 500)
    et_min = SAVED_POSTPROCESS.get("et_min_voxels", 50)

    for idx, m in enumerate(tqdm(metas, desc=desc, leave=False)):
        pred = predict_case_3d_sliding_window(model, m, patch_size=PATCH_SIZE, overlap=0.5)
        if apply_pp:
            pred = postprocess_volume_3d(pred, second_lcc_min_voxels=lcc_min, et_min_voxels=et_min)
        gt = np.load(m["seg_path"], mmap_mode="r")
        rows.append(brats_metrics(pred, gt))
        del pred, gt
        if (idx + 1) % 20 == 0:
            clear_memory()

    clear_memory()
    return {k: float(np.mean([r[k] for r in rows])) for k in ["WT", "TC", "ET", "Mean"]}

In [ ]:
class_weights = compute_class_weights(merged_train_metas).to(device)

train_ds = FastBraTS3DPatchDataset(
    merged_train_metas,
    patch_size=PATCH_SIZE,
    samples_per_volume=SAMPLES_PER_VOLUME,
    fg_prob=0.67,
    augment=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
    drop_last=True,
)

# Đánh giá nhanh mốc Baseline theta_0 trên Golden Validation 74 ca trước khi Finetune
model.load_state_dict(theta_0)
baseline_val_metrics = evaluate_golden_val_dice(model, val_metas_golden, apply_pp=True, desc="Baseline theta_0 (74 ca)")
print(
    f"[Baseline theta_0] WT={baseline_val_metrics['WT']:.4f} | "
    f"TC={baseline_val_metrics['TC']:.4f} | ET={baseline_val_metrics['ET']:.4f} | "
    f"Mean={baseline_val_metrics['Mean']:.4f}"
)

# Bắt đầu Fine-tune mô hình trên tập gộp 344 ca (294 BraTS2020 + 50 BraTS2021)
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-5)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

best_ft_mean = -1.0
best_ft_epoch = 0

print(f"\n=== [3/3] BẮT ĐẦU FINE-TUNE ({len(merged_train_metas)} ca | {FT_EPOCHS} epochs | LR={FT_LR}) ===")
for epoch in range(1, FT_EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    steps = 0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f"FT Epoch {epoch}/{FT_EPOCHS}", leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
            logits = model(x)
            loss = dice_ce_loss_3d(logits, y, class_weights=class_weights)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += float(loss.detach().item())
        steps += 1
        if steps % 50 == 0:
            pbar.set_postfix({"loss": f"{loss_sum / steps:.4f}"})

    epoch_min = (time.time() - t0) / 60.0
    mean_loss = loss_sum / max(steps, 1)
    clear_memory()

    if epoch % VAL_EVERY == 0 or epoch == FT_EPOCHS:
        val_m = evaluate_golden_val_dice(model, val_metas_golden, apply_pp=True, desc=f"Val Epoch {epoch}")
        msg = (
            f"Epoch {epoch:02d}/{FT_EPOCHS} | Loss={mean_loss:.4f} | Time={epoch_min:.2f}m | "
            f"WT={val_m['WT']:.4f} TC={val_m['TC']:.4f} ET={val_m['ET']:.4f} Mean={val_m['Mean']:.4f}"
        )
        if val_m["Mean"] > best_ft_mean:
            best_ft_mean = val_m["Mean"]
            best_ft_epoch = epoch
            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "best_mean_dice": best_ft_mean,
                "postprocess": SAVED_POSTPROCESS,
            }, str(FINETUNED_WEIGHTS_PATH))
            msg += " (BEST SAVED)"
        print(f"{msg} | {get_ram_status()}")
    else:
        print(f"Epoch {epoch:02d}/{FT_EPOCHS} | Loss={mean_loss:.4f} | Time={epoch_min:.2f}m | {get_ram_status()}")

del optimizer, scaler, train_loader, train_ds
clear_memory()

ft_ckpt = torch.load(str(FINETUNED_WEIGHTS_PATH), map_location="cpu", weights_only=False)
theta_ft = {k: v.detach().cpu().clone() for k, v in ft_ckpt["model_state_dict"].items()}
del ft_ckpt
clear_memory()
print(f"\nHoàn tất Fine-tune! Best checkpoint tại Epoch {best_ft_epoch} (Mean Dice: {best_ft_mean:.4f}). {get_ram_status()}")

Baseline theta_0 (74 ca):   0%|          | 0/74 [00:00<?, ?it/s]

[Baseline theta_0] WT=0.9026 | TC=0.8742 | ET=0.8004 | Mean=0.8591

=== [3/3] BẮT ĐẦU FINE-TUNE (344 ca | 15 epochs | LR=3e-05) ===


FT Epoch 1/15:   0%|          | 0/688 [00:00<?, ?it/s]

Epoch 01/15 | Loss=0.2664 | Time=3.33m | System RAM: 3.54/52.96 GB (8.6%) | GPU VRAM: 0.21 GB


FT Epoch 2/15:   0%|          | 0/688 [00:00<?, ?it/s]

Epoch 02/15 | Loss=0.2663 | Time=3.16m | System RAM: 3.40/52.96 GB (8.3%) | GPU VRAM: 0.21 GB


... (25 outputs left out)

Epoch 14/15 | Loss=0.2532 | Time=3.17m | System RAM: 3.40/52.96 GB (8.3%) | GPU VRAM: 0.21 GB


FT Epoch 15/15:   0%|          | 0/688 [00:00<?, ?it/s]

Val Epoch 15:   0%|          | 0/74 [00:00<?, ?it/s]

Epoch 15/15 | Loss=0.2517 | Time=3.18m | WT=0.9020 TC=0.8778 ET=0.7932 Mean=0.8577 | System RAM: 3.42/52.96 GB (8.4%) | GPU VRAM: 0.21 GB

Hoàn tất Fine-tune! Best checkpoint tại Epoch 10 (Mean Dice: 0.8603). System RAM: 3.44/52.96 GB (8.4%) | GPU VRAM: 0.17 GB


In [ ]:
PERCENTILES_LEVELS = np.linspace(0.01, 0.99, 99)

@torch.no_grad()
def compute_single_case_mc_cdfs_streaming(
    model: nn.Module,
    meta: dict,
    patch_size=(128, 128, 128),
    overlap=0.5,
    num_passes: int = 20,
    batch_t: int = 2
) -> Dict[str, np.ndarray]:
    """
    Chạy T=20 lượt MC Dropout theo từng mini-batch (batch_t=2) trên 1 ca bệnh duy nhất,
    tính ngay đường cong Soft Volumetric CDF [99] trên GPU và giải phóng toàn bộ tensor 4D.
    -> Tiết kiệm >99.9% System RAM (chỉ giữ lại mảng 99 số thực mỗi vùng, tuyệt đối không OOM).
    """
    enable_mc_dropout(model)
    image = np.load(meta["image_path"], mmap_mode="r")
    C, D, H, W = image.shape
    pD, pH, pW = patch_size

    step_d = max(1, int(pD * (1 - overlap)))
    step_h = max(1, int(pH * (1 - overlap)))
    step_w = max(1, int(pW * (1 - overlap)))

    steps_d = list(range(0, max(1, D - pD + 1), step_d))
    if steps_d[-1] + pD < D: steps_d.append(D - pD)
    steps_h = list(range(0, max(1, H - pH + 1), step_h))
    if steps_h[-1] + pH < H: steps_h.append(H - pH)
    steps_w = list(range(0, max(1, W - pW + 1), step_w))
    if steps_w[-1] + pW < W: steps_w.append(W - pW)

    wt_runs = torch.empty((num_passes, D, H, W), dtype=torch.float16, device=device)
    tc_runs = torch.empty((num_passes, D, H, W), dtype=torch.float16, device=device)
    et_runs = torch.empty((num_passes, D, H, W), dtype=torch.float16, device=device)

    for t_start in range(0, num_passes, batch_t):
        cur_b = min(batch_t, num_passes - t_start)
        prob_accum = torch.zeros((cur_b, 4, D, H, W), dtype=torch.float32, device=device)
        count_accum = torch.zeros((D, H, W), dtype=torch.float32, device=device)

        for sd in steps_d:
            for sh in steps_h:
                for sw in steps_w:
                    sub_vol = np.asarray(image[:, sd:sd+pD, sh:sh+pH, sw:sw+pW], dtype=np.float32)
                    x = torch.from_numpy(sub_vol).unsqueeze(0).repeat(cur_b, 1, 1, 1, 1).to(device)
                    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                        logits = model(x)
                        probs = torch.softmax(logits, dim=1)
                    prob_accum[:, :, sd:sd+pD, sh:sh+pH, sw:sw+pW] += probs
                    count_accum[sd:sd+pD, sh:sh+pH, sw:sw+pW] += 1.0
                    del x, logits, probs

        prob_accum /= torch.clamp(count_accum, min=1.0)[None, None, ...]
        et_runs[t_start:t_start+cur_b] = prob_accum[:, 1].half()
        tc_runs[t_start:t_start+cur_b] = (prob_accum[:, 1] + prob_accum[:, 2]).half()
        wt_runs[t_start:t_start+cur_b] = (prob_accum[:, 1] + prob_accum[:, 2] + prob_accum[:, 3]).half()
        del prob_accum, count_accum

    q_levels = torch.linspace(0.01, 0.99, 99, device=device)
    cdfs = {}
    for mod_name, runs_tensor in [("WT", wt_runs), ("TC", tc_runs), ("ET", et_runs)]:
        t_float = runs_tensor.float()
        valid_mask = (t_float.max(dim=0).values > 0.01)
        if not valid_mask.any():
            cdfs[mod_name] = np.zeros(99, dtype=np.float64)
        else:
            valid_voxels = t_float[:, valid_mask]
            q_voxels = torch.quantile(valid_voxels, q_levels, dim=0)
            vol_cdf = (q_voxels.sum(dim=1) * VOXEL_VOLUME_CM3).cpu().numpy().astype(np.float64)
            cdfs[mod_name] = np.maximum.accumulate(vol_cdf)
            del valid_voxels, q_voxels
        del t_float, valid_mask

    del wt_runs, tc_runs, et_runs
    return cdfs


def compute_conformal_scores_from_cdfs(cdfs_vols: np.ndarray, gt_vols: np.ndarray, modality: str) -> np.ndarray:
    scores = []
    for cdf_vol, gt_vol in zip(cdfs_vols, gt_vols):
        if modality == "ET" and gt_vol > 50.0:
            continue
        v_low_raw = cdf_vol[2]    # ~2.5%
        v_med = cdf_vol[49]       # 50%
        v_high_raw = cdf_vol[96]  # ~97.5%

        raw_half_width = max((v_high_raw - v_low_raw) / 2.0, 0.05 * v_med + 0.2)
        s = abs(gt_vol - v_med) / raw_half_width
        scores.append(s)
    return np.array(scores, dtype=np.float64)


def fit_conformal_scale(scores_train: np.ndarray, target_coverage: float = 0.95) -> float:
    N = len(scores_train)
    if N == 0:
        return 1.25
    q_val = min(1.0, np.ceil((N + 1) * target_coverage) / N)
    a_scale = float(np.quantile(scores_train, min(q_val, 0.999)))
    return max(1.0, a_scale)


def evaluate_calibrated_coverage_95(cdfs_vols: np.ndarray, gt_vols: np.ndarray, a_scale: float, modality: str) -> float:
    covered = 0
    valid_n = 0
    for cdf_vol, gt_vol in zip(cdfs_vols, gt_vols):
        if modality == "ET" and gt_vol > 50.0:
            continue
        valid_n += 1
        v_low_raw = cdf_vol[2]
        v_med = cdf_vol[49]
        v_high_raw = cdf_vol[96]

        raw_half_width = max((v_high_raw - v_low_raw) / 2.0, 0.05 * v_med + 0.2)
        ci_low = max(0.0, v_med - a_scale * raw_half_width)
        ci_high = v_med + a_scale * raw_half_width
        if ci_low <= gt_vol <= ci_high:
            covered += 1
    return float(covered / max(valid_n, 1))


def run_full_calibration_on_golden_val(
    model: nn.Module,
    val_metas: List[dict],
    dataset_label: str = "val_calibration_golden_74",
    save_json_path: Path = None
) -> Dict[str, Any]:
    all_cdfs = {"WT": [], "TC": [], "ET": []}
    all_gts = {"WT": [], "TC": [], "ET": []}

    for idx, m in enumerate(tqdm(val_metas, desc=f"MC Dropout Calibration ({dataset_label})", leave=False)):
        gt_seg = np.load(m["seg_path"], mmap_mode="r")
        all_gts["WT"].append(float(np.sum(gt_seg > 0) * VOXEL_VOLUME_CM3))
        all_gts["TC"].append(float(np.sum(np.isin(gt_seg, [1, 2])) * VOXEL_VOLUME_CM3))
        all_gts["ET"].append(float(np.sum(gt_seg == 1) * VOXEL_VOLUME_CM3))
        del gt_seg

        case_cdfs = compute_single_case_mc_cdfs_streaming(
            model, m, patch_size=PATCH_SIZE, overlap=0.5, num_passes=MC_DROPOUT_PASSES, batch_t=2
        )
        for mod in ["WT", "TC", "ET"]:
            all_cdfs[mod].append(case_cdfs[mod])
        del case_cdfs

        if (idx + 1) % 10 == 0:
            clear_memory()

    clear_memory()

    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    final_factors = {}
    folds_details = {"WT": [], "TC": [], "ET": []}
    mean_cov_summary = {}

    for mod in ["WT", "TC", "ET"]:
        cdfs_arr = np.array(all_cdfs[mod], dtype=np.float64)
        gts_arr = np.array(all_gts[mod], dtype=np.float64)

        fold_covs = []
        for fold_idx, (tr_idx, te_idx) in enumerate(kf.split(cdfs_arr), start=1):
            tr_scores = compute_conformal_scores_from_cdfs(cdfs_arr[tr_idx], gts_arr[tr_idx], modality=mod)
            a_fold = fit_conformal_scale(tr_scores, target_coverage=0.95)
            cov_95 = evaluate_calibrated_coverage_95(cdfs_arr[te_idx], gts_arr[te_idx], a_fold, modality=mod)
            fold_covs.append(cov_95)
            folds_details[mod].append({
                "fold": fold_idx,
                "train_cases": int(len(tr_idx)),
                "val_cases": int(len(te_idx)),
                "slope": round(float(a_fold), 4),
                "coverage_95": round(float(cov_95), 4),
            })

        all_scores = compute_conformal_scores_from_cdfs(cdfs_arr, gts_arr, modality=mod)
        a_final = fit_conformal_scale(all_scores, target_coverage=0.95)
        full_cov = evaluate_calibrated_coverage_95(cdfs_arr, gts_arr, a_final, modality=mod)

        final_factors[f"a_{mod.lower()}"] = round(float(a_final), 4)
        mean_cov_summary[mod] = {
            "cv_mean_coverage_95": round(float(np.mean(fold_covs)), 4),
            "full_val_coverage_95": round(float(full_cov), 4),
        }

    cache_payload = {
        "calibration_factors": final_factors,
        "validation_details": {
            "dataset": dataset_label,
            "total_cases": int(len(val_metas)),
            "mc_dropout_passes": int(MC_DROPOUT_PASSES),
            "method": "Conformal Volumetric Quantile Scaling (3-Fold CV)",
            "coverage_summary": mean_cov_summary,
            "folds": folds_details,
        }
    }

    if save_json_path is not None:
        with open(save_json_path, "w", encoding="utf-8") as f:
            json.dump(cache_payload, f, indent=2)

    return cache_payload

In [ ]:
# ==============================================================================
# CELL 1: CONTINUAL OPCM MERGING (LOW-RANK SVD) + CALIBRATION TRÊN TẬP VAL 74 CA
# ==============================================================================
def opcm_project_matrix_lowrank(delta_W_new: torch.Tensor, delta_W_ref: torch.Tensor, rank: int = 16) -> torch.Tensor:
    """
    Chiếu trực giao sử dụng Low-Rank SVD (torch.svd_lowrank) theo mẫu brainfm_continual:
    - Tiết kiệm 95% System RAM so với SVD toàn phần (tránh hoàn toàn Colab OOM)
    - Triệt tiêu thành phần giao thoa trên 'rank' trục chính của không gian tham chiếu
    """
    m, n = delta_W_ref.shape
    q = min(rank, min(m, n))
    if q < 1:
        return delta_W_new

    try:
        U_sub, S_sub, V_sub = torch.svd_lowrank(delta_W_ref, q=q, niter=2)
    except Exception:
        return delta_W_new

    M = torch.matmul(torch.matmul(U_sub.T, delta_W_new), V_sub)
    diag_elements = torch.diag(M)
    interfere = torch.matmul(torch.matmul(U_sub, torch.diag(diag_elements)), V_sub.T)
    projected = delta_W_new - interfere
    del U_sub, S_sub, V_sub, M, diag_elements, interfere
    return projected


def continual_opcm_merge_3dunet(
    theta_base: Dict[str, torch.Tensor],
    theta_finetuned: Dict[str, torch.Tensor],
    rank: int = 16,
    selective_layers: bool = True,
    alpha_scale: float = 0.5
) -> Dict[str, torch.Tensor]:
    merged = {}
    with torch.no_grad():
        for key in list(theta_base.keys()):
            w_0 = theta_base[key].float()
            w_ft = theta_finetuned[key].float()

            if "head" in key:
                w_m = (1.0 - alpha_scale) * w_0 + alpha_scale * w_ft
                merged[key] = w_m.to(theta_base[key].dtype)
                continue

            delta_new = w_ft - w_0
            is_conv_or_linear = (delta_new.dim() >= 2) and ("weight" in key) and ("norm" not in key)
            is_shallow = ("stem" in key) or ("enc1" in key)
            should_project = is_conv_or_linear and (not selective_layers or not is_shallow)

            if should_project:
                orig_shape = delta_new.shape
                m = orig_shape[0]
                w0_2d = w_0.view(m, -1)
                delta_new_2d = delta_new.view(m, -1)

                proj_delta_2d = opcm_project_matrix_lowrank(delta_new_2d, w0_2d, rank=rank)
                proj_delta = proj_delta_2d.view(orig_shape)
                w_m = w_0 + alpha_scale * proj_delta
                del w0_2d, delta_new_2d, proj_delta_2d, proj_delta
            else:
                w_m = w_0 + alpha_scale * delta_new

            merged[key] = w_m.to(theta_base[key].dtype)
            del w_0, w_ft, delta_new, w_m

    trim_os_memory()
    return merged


print("=== [CELL 1] CHẠY CONTINUAL OPCM MERGING & CALIBRATION ===")
theta_opcm = continual_opcm_merge_3dunet(
    theta_base=theta_0,
    theta_finetuned=theta_ft,
    rank=16,
    selective_layers=True,
    alpha_scale=0.5
)
torch.save({
    "model_state_dict": theta_opcm,
    "method": "OPCM (Low-Rank SVD, rank=16)",
    "postprocess": SAVED_POSTPROCESS,
}, str(OPCM_WEIGHTS_PATH))

model.load_state_dict(theta_opcm)
del theta_opcm
clear_memory()

# 1. Đánh giá Dice trên Golden Validation 74 ca
opcm_dice = evaluate_golden_val_dice(model, val_metas_golden, apply_pp=True, desc="OPCM Golden Val Dice (74 ca)")

# 2. Chạy Calibration MC Dropout (T=20) trên Golden Validation 74 ca
OPCM_CALIB_PATH = Path("/content/calibration_cache_opcm.json")
opcm_calib = run_full_calibration_on_golden_val(
    model=model,
    val_metas=val_metas_golden,
    dataset_label="opcm_golden_val_74",
    save_json_path=OPCM_CALIB_PATH
)

opcm_factors = opcm_calib["calibration_factors"]
opcm_cov = opcm_calib["validation_details"]["coverage_summary"]

print("\n" + "=" * 72)
print("KẾT QUẢ CELL 1 — OPCM MERGING TRÊN GOLDEN VALIDATION (74 CA):")
print("=" * 72)
print(f"Dice Score   | WT: {opcm_dice['WT']:.4f} | TC: {opcm_dice['TC']:.4f} | ET: {opcm_dice['ET']:.4f} | Mean: {opcm_dice['Mean']:.4f}")
print(f"Calib Scale  | a_wt: {opcm_factors['a_wt']:.4f} | a_tc: {opcm_factors['a_tc']:.4f} | a_et: {opcm_factors['a_et']:.4f}")
print(
    f"Coverage 95% | WT: {opcm_cov['WT']['cv_mean_coverage_95']*100:.2f}% (Full: {opcm_cov['WT']['full_val_coverage_95']*100:.2f}%) | "
    f"TC: {opcm_cov['TC']['cv_mean_coverage_95']*100:.2f}% (Full: {opcm_cov['TC']['full_val_coverage_95']*100:.2f}%) | "
    f"ET: {opcm_cov['ET']['cv_mean_coverage_95']*100:.2f}% (Full: {opcm_cov['ET']['full_val_coverage_95']*100:.2f}%)"
)
print(f"Đã lưu calibration_cache OPCM tại: {OPCM_CALIB_PATH} | {get_ram_status()}")

=== [CELL 1] CHẠY CONTINUAL OPCM MERGING & CALIBRATION ===


OPCM Golden Val Dice (74 ca):   0%|          | 0/74 [00:00<?, ?it/s]

MC Dropout Calibration (opcm_golden_val_74):   0%|          | 0/74 [00:00<?, ?it/s]


KẾT QUẢ CELL 1 — OPCM MERGING TRÊN GOLDEN VALIDATION (74 CA):
Dice Score   | WT: 0.9036 | TC: 0.8771 | ET: 0.7914 | Mean: 0.8574
Calib Scale  | a_wt: 3.9789 | a_tc: 5.8083 | a_et: 8.5595
Coverage 95% | WT: 97.28% (Full: 97.30%) | TC: 95.94% (Full: 97.30%) | ET: 97.15% (Full: 97.06%)
Đã lưu calibration_cache OPCM tại: /content/calibration_cache_opcm.json | System RAM: 3.63/52.96 GB (8.8%) | GPU VRAM: 0.17 GB


In [ ]:
# ==============================================================================
# CELL 2: QUÉT HỆ SỐ TRỘN WiSE-FT (lambda in {0.1, 0.2, 0.3, 0.5, 1.0})
# SO SÁNH DICE (WT, TC, ET) + KIỂM TRA COVERAGE_95 SAU CALIBRATION -> CHỐT PRODUCTION
# ==============================================================================
WISE_FT_LAMBDAS = [0.1, 0.2, 0.3, 0.5, 1.0]
wise_ft_results = []

print("=== [CELL 2] BẮT ĐẦU QUÉT WiSE-FT TRÊN GOLDEN VALIDATION (74 CA) ===")
for lam in WISE_FT_LAMBDAS:
    print(f"\n--- Đang đánh giá WiSE-FT với lambda = {lam:.1f} ---")
    with torch.no_grad():
        theta_merged = {
            k: ((1.0 - lam) * theta_0[k].float() + lam * theta_ft[k].float()).to(theta_0[k].dtype)
            for k in theta_0.keys()
        }

    model.load_state_dict(theta_merged)
    del theta_merged
    clear_memory()

    # 1. Đo Dice (WT, TC, ET, Mean) trên 74 ca Golden Validation
    dice_res = evaluate_golden_val_dice(
        model, val_metas_golden, apply_pp=True, desc=f"WiSE-FT (lam={lam}) Dice"
    )

    # 2. Chạy lại hiệu chuẩn calibration_cache và kiểm tra coverage_95
    lam_calib_path = Path(f"/content/calibration_cache_wise_ft_lam_{str(lam).replace('.', '_')}.json")
    calib_res = run_full_calibration_on_golden_val(
        model=model,
        val_metas=val_metas_golden,
        dataset_label=f"wise_ft_lam_{lam}_golden_74",
        save_json_path=lam_calib_path
    )

    factors = calib_res["calibration_factors"]
    cov_sum = calib_res["validation_details"]["coverage_summary"]
    mean_cv_cov = float(np.mean([
        cov_sum["WT"]["cv_mean_coverage_95"],
        cov_sum["TC"]["cv_mean_coverage_95"],
        cov_sum["ET"]["cv_mean_coverage_95"],
    ]))

    gate_pass = (
        dice_res["WT"] >= baseline_val_metrics["WT"] - 0.005 and
        dice_res["TC"] >= baseline_val_metrics["TC"] - 0.005 and
        dice_res["ET"] >= baseline_val_metrics["ET"] - 0.005 and
        mean_cv_cov >= 0.88
    )

    wise_ft_results.append({
        "lambda": lam,
        "WT": dice_res["WT"],
        "TC": dice_res["TC"],
        "ET": dice_res["ET"],
        "Mean_Dice": dice_res["Mean"],
        "a_wt": factors["a_wt"],
        "a_tc": factors["a_tc"],
        "a_et": factors["a_et"],
        "cov95_WT": cov_sum["WT"]["cv_mean_coverage_95"],
        "cov95_TC": cov_sum["TC"]["cv_mean_coverage_95"],
        "cov95_ET": cov_sum["ET"]["cv_mean_coverage_95"],
        "Mean_Cov95": mean_cv_cov,
        "gate_pass": gate_pass,
        "calib_path": str(lam_calib_path),
    })

    print(
        f"[lambda={lam:.1f}] Dice: WT={dice_res['WT']:.4f}, TC={dice_res['TC']:.4f}, ET={dice_res['ET']:.4f} (Mean={dice_res['Mean']:.4f}) | "
        f"Calib a=({factors['a_wt']:.2f}, {factors['a_tc']:.2f}, {factors['a_et']:.2f}) | "
        f"Cov95=({cov_sum['WT']['cv_mean_coverage_95']*100:.1f}%, {cov_sum['TC']['cv_mean_coverage_95']*100:.1f}%, {cov_sum['ET']['cv_mean_coverage_95']*100:.1f}%) | "
        f"Gate={'PASS' if gate_pass else 'FAIL'} | {get_ram_status()}"
    )

# ==============================================================================
# BẢNG SO SÁNH TỔNG HỢP & CHỐT LAMBDA TỐI ƯU CHO PRODUCTION
# ==============================================================================
print("\n" + "=" * 115)
print("BẢNG SO SÁNH HIỆU NĂNG & HIỆU CHUẨN TRÊN GOLDEN VALIDATION (74 CA)")
print("=" * 115)
header = (
    f"{'Method / Lambda':<18} | {'WT Dice':<8} | {'TC Dice':<8} | {'ET Dice':<8} | {'Mean Dice':<9} | "
    f"{'a (WT/TC/ET)':<20} | {'Cov95 (WT/TC/ET)':<22} | {'Gate':<6}"
)
print(header)
print("-" * 115)

print(
    f"{'Baseline (lam=0.0)':<18} | {baseline_val_metrics['WT']:<8.4f} | {baseline_val_metrics['TC']:<8.4f} | "
    f"{baseline_val_metrics['ET']:<8.4f} | {baseline_val_metrics['Mean']:<9.4f} | "
    f"{'N/A':<20} | {'N/A':<22} | {'REF':<6}"
)

print(
    f"{'OPCM (Cell 1)':<18} | {opcm_dice['WT']:<8.4f} | {opcm_dice['TC']:<8.4f} | "
    f"{opcm_dice['ET']:<8.4f} | {opcm_dice['Mean']:<9.4f} | "
    f"{opcm_factors['a_wt']:.2f}/{opcm_factors['a_tc']:.2f}/{opcm_factors['a_et']:.2f}   | "
    f"{opcm_cov['WT']['cv_mean_coverage_95']*100:5.1f}%/{opcm_cov['TC']['cv_mean_coverage_95']*100:5.1f}%/{opcm_cov['ET']['cv_mean_coverage_95']*100:5.1f}%   | "
    f"{'OPCM':<6}"
)
print("-" * 115)

for r in wise_ft_results:
    a_str = f"{r['a_wt']:.2f}/{r['a_tc']:.2f}/{r['a_et']:.2f}"
    c_str = f"{r['cov95_WT']*100:5.1f}%/{r['cov95_TC']*100:5.1f}%/{r['cov95_ET']*100:5.1f}%"
    print(
        f"{'WiSE-FT lam=' + str(r['lambda']):<18} | {r['WT']:<8.4f} | {r['TC']:<8.4f} | "
        f"{r['ET']:<8.4f} | {r['Mean_Dice']:<9.4f} | {a_str:<20} | {c_str:<22} | "
        f"{('PASS' if r['gate_pass'] else 'FAIL'):<6}"
    )
print("=" * 115)

passed_candidates = [r for r in wise_ft_results if r["gate_pass"]]
if len(passed_candidates) > 0:
    best_candidate = max(passed_candidates, key=lambda x: (x["Mean_Dice"], x["Mean_Cov95"]))
else:
    best_candidate = max(wise_ft_results, key=lambda x: x["Mean_Dice"])

BEST_LAMBDA = best_candidate["lambda"]
print(f"\n==> CHỐT HỆ SỐ TỐI ƯU CHO PRODUCTION: lambda* = {BEST_LAMBDA}")
print(
    f"    - Golden Val Dice: WT={best_candidate['WT']:.4f} | TC={best_candidate['TC']:.4f} | "
    f"ET={best_candidate['ET']:.4f} | Mean={best_candidate['Mean_Dice']:.4f}"
)
print(
    f"    - Calibration Factors: a_wt={best_candidate['a_wt']:.4f}, "
    f"a_tc={best_candidate['a_tc']:.4f}, a_et={best_candidate['a_et']:.4f}"
)
print(
    f"    - Coverage 95% (3-Fold CV): WT={best_candidate['cov95_WT']*100:.2f}%, "
    f"TC={best_candidate['cov95_TC']*100:.2f}%, ET={best_candidate['cov95_ET']*100:.2f}%"
)

with torch.no_grad():
    prod_state_dict = {
        k: ((1.0 - BEST_LAMBDA) * theta_0[k].float() + BEST_LAMBDA * theta_ft[k].float()).to(theta_0[k].dtype)
        for k in theta_0.keys()
    }

torch.save({
    "model_state_dict": prod_state_dict,
    "architecture": "3D Residual U-Net",
    "base_features": BASE_FEATURES,
    "bottleneck": BOTTLENECK_FEATURES,
    "merging_method": "WiSE-FT",
    "wise_ft_lambda": BEST_LAMBDA,
    "golden_val_metrics": {
        "WT": best_candidate["WT"],
        "TC": best_candidate["TC"],
        "ET": best_candidate["ET"],
        "Mean": best_candidate["Mean_Dice"],
    },
    "postprocess": SAVED_POSTPROCESS,
    "class_order": ["BG", "ET", "NC", "OE"],
}, str(PROD_WEIGHTS_PATH))

shutil.copyfile(best_candidate["calib_path"], str(PROD_CALIB_PATH))
del prod_state_dict
clear_memory()

print(f"\n[PRODUCTION READY] Đã xuất trọng số tại:         {PROD_WEIGHTS_PATH}")
print(f"[PRODUCTION READY] Đã xuất calibration_cache tại: {PROD_CALIB_PATH}")
print(get_ram_status())

=== [CELL 2] BẮT ĐẦU QUÉT WiSE-FT TRÊN GOLDEN VALIDATION (74 CA) ===

--- Đang đánh giá WiSE-FT với lambda = 0.1 ---


WiSE-FT (lam=0.1) Dice:   0%|          | 0/74 [00:00<?, ?it/s]

MC Dropout Calibration (wise_ft_lam_0.1_golden_74):   0%|          | 0/74 [00:00<?, ?it/s]

[lambda=0.1] Dice: WT=0.9030, TC=0.8749, ET=0.8016 (Mean=0.8598) | Calib a=(4.05, 5.97, 8.65) | Cov95=(97.2%, 95.9%, 97.2%) | Gate=PASS | System RAM: 3.13/52.96 GB (7.8%) | GPU VRAM: 0.17 GB

--- Đang đánh giá WiSE-FT với lambda = 0.2 ---


WiSE-FT (lam=0.2) Dice:   0%|          | 0/74 [00:00<?, ?it/s]

MC Dropout Calibration (wise_ft_lam_0.2_golden_74):   0%|          | 0/74 [00:00<?, ?it/s]

... (6 outputs left out)

[lambda=0.5] Dice: WT=0.9051, TC=0.8773, ET=0.7918 (Mean=0.8581) | Calib a=(3.87, 6.28, 9.52) | Cov95=(97.2%, 95.9%, 96.9%) | Gate=FAIL | System RAM: 3.13/52.96 GB (7.8%) | GPU VRAM: 0.17 GB

--- Đang đánh giá WiSE-FT với lambda = 1.0 ---


WiSE-FT (lam=1.0) Dice:   0%|          | 0/74 [00:00<?, ?it/s]

MC Dropout Calibration (wise_ft_lam_1.0_golden_74):   0%|          | 0/74 [00:00<?, ?it/s]

[lambda=1.0] Dice: WT=0.9069, TC=0.8775, ET=0.7963 (Mean=0.8603) | Calib a=(3.57, 6.15, 7.53) | Cov95=(97.2%, 97.3%, 97.1%) | Gate=PASS | System RAM: 3.13/52.96 GB (7.8%) | GPU VRAM: 0.17 GB

BẢNG SO SÁNH HIỆU NĂNG & HIỆU CHUẨN TRÊN GOLDEN VALIDATION (74 CA)
Method / Lambda    | WT Dice  | TC Dice  | ET Dice  | Mean Dice | a (WT/TC/ET)         | Cov95 (WT/TC/ET)       | Gate  
-------------------------------------------------------------------------------------------------------------------
Baseline (lam=0.0) | 0.9026   | 0.8742   | 0.8004   | 0.8591    | N/A                  | N/A                    | REF   
OPCM (Cell 1)      | 0.9036   | 0.8771   | 0.7914   | 0.8574    | 3.98/5.81/8.56   |  97.3%/ 95.9%/ 97.2%   | OPCM  
-------------------------------------------------------------------------------------------------------------------
WiSE-FT lam=0.1    | 0.9030   | 0.8749   | 0.8016   | 0.8598    | 4.05/5.97/8.65       |  97.2%/ 95.9%/ 97.2%   | PASS  
WiSE-FT lam=0.2    | 0.9036  